# Notebook 03 — Baseline Models

**Goal:** Train Logistic Regression and LightGBM (with Optuna tuning) as baselines.

These set the bar — if deep learning can't beat well-tuned LightGBM, it's not worth the complexity.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import optuna
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report

from src.utils import set_seed, evaluate_classifier, plot_confusion_matrix
from src.models import build_logistic_regression, build_lightgbm
from src.data_loader import train_val_test_split

set_seed(42)
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline

RESULTS_DIR = Path('../results')
(RESULTS_DIR / 'plots').mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'tables').mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'models').mkdir(parents=True, exist_ok=True)

## 1. Load Processed Features

In [ ]:
df = pd.read_parquet('../data/processed/features.parquet')
print(f'Loaded: {df.shape}')

# Use k=3 horizon labels (30 events)
label_col = 'label'
feature_cols = [c for c in df.columns if not c.startswith('label')]

X = df[feature_cols].values.astype(np.float32)
y = df[label_col].values.astype(np.int64)

# Handle NaN/Inf
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

print(f'Features: {X.shape}, Labels: {y.shape}')
print(f'Classes: {np.bincount(y)}')

## 2. Chronological Train/Val/Test Split

In [ ]:
splits = train_val_test_split(X, y, train_ratio=0.7, val_ratio=0.15)
X_train, y_train = splits['X_train'], splits['y_train']
X_val, y_val = splits['X_val'], splits['y_val']
X_test, y_test = splits['X_test'], splits['y_test']

# Standardize features (fit on train only!)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

joblib.dump(scaler, RESULTS_DIR / 'models' / 'scaler.joblib')
print('Scaler saved.')

## 3. Logistic Regression Baseline

In [ ]:
lr_model = build_logistic_regression(C=1.0)
lr_model.fit(X_train_s, y_train)

lr_preds = lr_model.predict(X_test_s)
lr_metrics = evaluate_classifier(y_test, lr_preds, title='Logistic Regression')
plot_confusion_matrix(y_test, lr_preds, title='Logistic Regression',
                     save_path=str(RESULTS_DIR / 'plots' / 'cm_logistic.png'))

joblib.dump(lr_model, RESULTS_DIR / 'models' / 'logistic_regression.joblib')

## 4. LightGBM with Optuna Hyperparameter Tuning

In [ ]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    }
    model = build_lightgbm(params)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb_cb])
    preds = model.predict(X_val)
    return f1_score(y_val, preds, average='macro')

import lightgbm as lgb
lgb_cb = lgb.early_stopping(50, verbose=False)

# Run Optuna (30 trials — fast enough on CPU)
study = optuna.create_study(direction='maximize', study_name='lgbm_tuning')
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f'\nBest F1 (macro): {study.best_value:.4f}')
print(f'Best params: {study.best_params}')

In [ ]:
# Train final LightGBM with best params
best_lgbm = build_lightgbm(study.best_params)
best_lgbm.fit(X_train, y_train)

lgbm_preds = best_lgbm.predict(X_test)
lgbm_metrics = evaluate_classifier(y_test, lgbm_preds, title='LightGBM (Optuna)')
plot_confusion_matrix(y_test, lgbm_preds, title='LightGBM (Optuna)',
                     save_path=str(RESULTS_DIR / 'plots' / 'cm_lightgbm.png'))

joblib.dump(best_lgbm, RESULTS_DIR / 'models' / 'lightgbm_best.joblib')

## 5. Feature Importance (LightGBM)

In [ ]:
from src.utils import plot_feature_importance

importances = best_lgbm.feature_importances_
plot_feature_importance(
    feature_cols, importances, top_n=20,
    title='LightGBM Feature Importance (Top 20)',
    save_path=str(RESULTS_DIR / 'plots' / 'lgbm_feature_importance.png')
)

## 6. Results Summary

In [ ]:
results_df = pd.DataFrame([
    {'Model': 'Logistic Regression', **lr_metrics},
    {'Model': 'LightGBM (Optuna)', **lgbm_metrics},
])
results_df.to_csv(RESULTS_DIR / 'tables' / 'baseline_results.csv', index=False)
print('\n=== Baseline Results ===')
print(results_df.to_string(index=False))
print(f'\nSaved to: results/tables/baseline_results.csv')

## Summary

- Logistic Regression provides a simple baseline
- LightGBM with Optuna tuning typically far outperforms LR
- Feature importance reveals OFI and spread as top predictors

**Next:** Notebook 04 — LSTM & GRU